# tables statisticsRun order matters only within a notebook; each cell below is the code that produced the corresponding table or figure.

In [ ]:
import argparse, os, pathlib, syssys.path.insert(0, str(pathlib.Path.cwd().parent / "mospr"))import paths as _pTABLES = _p.TABLESns = lambda **kw: argparse.Namespace(**kw)RESULTS = _p.RESULTS

## Table 1 with 95% CI, fold SD and paired p against MoSPRCode: make_table1_variants (this notebook)

In [ ]:
import argparse, pathlibimport numpy as npimport pandas as pdfrom scipy import statsR = pathlib._p.ROOT                                                 import osimport sys as _sys, pathlib as _pl_sys.path.insert(0, str(_pl.Path(__file__).resolve().parent))import paths as _pSRC = os.environ.get("MOSPR_TABLE_SRC", str(_p.TABLES / "per_cohort"))TCRIT = stats.t.ppf(0.975, 3)OURS = "MoSPR (ours)"COH = ["BRCA", "KIRC", "LUAD"]                                                                                                                                                                                                    ROWS = [("AbMIL max-pool", r"Max~\citep{wang2018revisiting}"),        ("AbMIL mean-pool", r"Mean~\citep{wang2018revisiting}"),        ("AbMIL", r"AbMIL~\citep{ilse2018abmil}"),        ("HE2RNA", r"HE2RNA~\citep{schmauch2020he2rna}"),        ("AbReg", r"AbReg~\citep{graziani2022abreg}"),        ("tRNAformer", r"tRNAformer~\citep{alsaafin2022trnaformer}"),        ("ILRA", r"ILRA~\citep{xiang2023ilra}"),        ("S4Model", r"S4MIL~\citep{fillioux2023s4mil}"),        ("MambaMIL", r"MambaMIL~\citep{yang2024mambamil}"),        ("SRMambaMIL", r"SRMambaMIL~\citep{yang2024mambamil}"),        ("MOSBY", r"MOSBY~\citep{senbabaoglu2024mosby}"),        ("SEQUOIA VIS", r"SEQUOIA VIS~\citep{pizurica2024sequoia}"),        ("2DMamba", r"2DMamba~\citep{zhang2025twodmamba}"),        ("CPNN", r"CPNN~\citep{nishimura2026cpnn}"),        ("__MID__", None),        (OURS, r"\textbf{MoSPR (ours)}")]def perfold(c, tag=""):    d = R / SRC / c / "results"    g = pd.read_csv(d / "baselines_per_fold.csv")    sc = [x for x in g.columns if x.upper() == "SCC"][0]    pc = [x for x in g.columns if x.upper() == "PCC"][0]    out = {m: (s.sort_values("fold")[pc].values, s.sort_values("fold")[sc].values)           for m, s in g.groupby("model")}    r = d / "cpnn_rescored.csv"    if r.exists() and r.stat().st_size > 20:        t = pd.read_csv(r).sort_values("fold")        out["CPNN"] = (t.PCC.values, t.SCC.values)    m = pd.read_csv(RESULTS / f"cohort_{c}_fpsplit{tag}_tv_ours.csv")    m = m[(m.primary) & (m.space == "gene")].sort_values("fold")    out[OURS] = (m.test_PCC.values, m.test_SCC.values)    return {k: (np.asarray(a, float), np.asarray(b, float))            for k, (a, b) in out.items() if len(a) == 4}def fmt_p(p):    return r"\textless.001" if p < 1e-3 else f"{p:.3f}"HEADS = {    "ci":  ("Table 1 - mean +- 95% CI",            "% Values are the 4-fold mean +- 95% CI, taken from the t distribution across folds\n"            "% (df=3, t=3.182). The repeated unit is the fold, not the gene: genes are\n"            "% strongly correlated, so intervals over genes come out unrealistically narrow.\n"            "% Note: intervals are computed per model. Overlapping intervals of two models do\n"            "% not mean there is no difference: every model sees the same folds (same test slides),\n"            "% so comparisons between models must be paired (see the p-value variant).\n"),    "std": ("Table 1 - mean (fold SD)",            "% Parentheses hold the 4-fold standard deviation (ddof=1), i.e. spread, not an interval.\n"            "% With only four folds the standard deviation itself is an unstable estimate.\n"),    "p":   ("Table 1 - mean and p against MoSPR",            "% p is for the fold-paired difference in gene SCC between MoSPR and that model,\n"            "% (n=4 folds, df=3). The null hypothesis is that the mean fold-wise SCC difference\n"            "% between the two models is zero. All models see the same four folds, so the runs are\n"            "% paired: treating them as independent samples would count fold difficulty as\n"            "% noise and lose power.\n"            "% No multiple-comparison correction (14 comparisons per cohort).\n"),    "all": ("Table 1 - mean +- 95% CI (SD), p against MoSPR",            "% Each cell is mean +- 95% CI (t across folds, df=3), with the standard deviation in parentheses.\n"            "% p is the two-sided paired t-test on the fold-wise gene SCC difference against MoSPR\n"            "% (n=4, df=3). No multiple-comparison correction.\n"),}def build(kind, data, alpha, exclude=()):    withp = kind in ("p", "all")    ncol = 3 * (3 if withp else 2)    spec = "l" + "c" * ncol    sub = ("PCC & SCC & $p$" if withp else "PCC & SCC")    span = 3 if withp else 2    heads, cmids, i = [], [], 2    for c in COH:        heads.append(r"\multicolumn{%d}{c}{%s}" % (span, c))        cmids.append(r"\cmidrule(lr){%d-%d}" % (i, i + span - 1)); i += span    best = {(c, j): max(v[j].mean() for v in data[c].values()) for c in COH for j in (0, 1)}    L = []    for key, lab in [r for r in ROWS if r[0] not in exclude]:        if key == "__MID__":            L.append(r"\midrule"); continue        cells = []        for c in COH:            d = data[c]            for j in (0, 1):                if key not in d:                    cells.append("--"); continue                v = d[key][j]; mu = v.mean()                sd = v.std(ddof=1); hw = TCRIT * sd / 2.0                if kind == "ci":                    s = f"{mu:.3f}\\,{{\\tiny$\\pm$}}{hw:.3f}"                elif kind == "std":                    s = f"{mu:.3f}\\,{{\\tiny({sd:.3f})}}"                elif kind == "p":                    s = f"{mu:.3f}"                else:                    s = f"{mu:.3f}\\,{{\\tiny$\\pm$}}{hw:.3f}\\,{{\\tiny({sd:.3f})}}"                cells.append(r"\textbf{" + s + "}" if abs(mu - best[(c, j)]) < 1e-12 else s)            if withp:                if key == OURS or key not in d:                    cells.append("--")                else:                    _, p = stats.ttest_rel(d[OURS][1], d[key][1])                    cells.append(fmt_p(p))        L.append(f"{lab} & " + " & ".join(cells) + r"\\")    title, note = HEADS[kind]    hdr = (f"% {title}\n"           f"% Setting: filtered patches, patient-level split, train+val fine-tuning, seed 2021\n"           f"% K=8 · d=512 · tau=0 · ALPHA_SMOOTH={alpha}\n" + note + bis)    return (hdr + r"\begin{tabular}{" + spec + "}\n" + r"\toprule" + "\n"            + "& " + " & ".join(heads) + r"\\" + "\n"            + "".join(cmids) + "\n"            + "Method & " + " & ".join([sub] * 3) + r"\\" + "\n"            + r"\midrule" + "\n" + "\n".join(L) + "\n"            + r"\bottomrule" + "\n" + r"\end{tabular}")def main(a):    tag = "" if a.alpha == "0" else "_a1"    data = {c: perfold(c, tag) for c in COH}    out = R / a.out; out.mkdir(parents=True, exist_ok=True)    for kind in ("ci", "std", "p", "all"):        f = out / f"table1_gene_{kind}.tex"        f.write_text(build(kind, data, a.alpha, a.exclude))        print(f"  {f}")

## Appendix: fold mean / 95% CI / SD per methodCode: stats_table (this notebook)

In [ ]:
import argparsefrom pathlib import Pathimport numpy as npimport pandas as pdfrom scipy import statsROOT = _p.ROOT                                     import osimport sys as _sys, pathlib as _pl_sys.path.insert(0, str(_pl.Path(__file__).resolve().parent))import paths as _pSRC = Path(os.environ.get("MOSPR_STATS_SRC", _p.TABLES / "tables/final/alpha0/source"))GENE = Path(os.environ.get("MOSPR_STATS_GENE", _p.TABLES / "tables/refit/refit_per_fold_gene.csv"))OUT = Path(os.environ.get("MOSPR_STATS_OUT", _p.TABLES / "tables/statistics"))                                                    RENAME = {"AbMIL max-pool": "Max", "AbMIL mean-pool": "Mean", "S4Model": "S4MIL"}ORDER = ["Max", "Mean", "AbMIL", "HE2RNA", "AbReg", "tRNAformer", "ILRA", "S4MIL",         "MambaMIL", "SRMambaMIL", "MOSBY", "SEQUOIA VIS", "2DMamba",         "CPNN", "MoSPR (ours)"]COLLECTIONS = ["hallmark", "gobp", "kegg"]                                           PRETTY = {"gene PCC": "PCC", "gene SCC": "SCC",          "hallmark SCC (median)": "Hallmark", "gobp SCC (median)": "GO-BP",          "kegg SCC (median)": "KEGG"}def load_long(cohorts):    rows = []    g = pd.read_csv(GENE)    g = g[g.stage == "stage2_refit_trainval"].replace({"model": RENAME})    for _, r in g.iterrows():        if r.cohort not in cohorts:            continue        rows += [{"cohort": r.cohort, "model": r.model, "metric": f"gene {m}",                  "fold": int(r.fold), "value": r[m]} for m in ("PCC", "SCC")]    for c in cohorts:        p = pd.read_csv(SRC / f"pathway_all_{c}_fpsplit_tv.csv").replace({"model": RENAME})        for _, r in p[p.collection.isin(COLLECTIONS)].iterrows():            rows.append({"cohort": c, "model": r.model,                         "metric": f"{r.collection} SCC (median)",                         "fold": int(r.fold), "value": r.SCC_median})    d = pd.DataFrame(rows)    return d[~d.model.isin(EXCLUDE)].reset_index(drop=True)def summarise(long):    out = []    for (c, m, met), g in long.groupby(["cohort", "model", "metric"]):        v = g.sort_values("fold").value.values.astype(float)        n = len(v)        sd = v.std(ddof=1) if n > 1 else np.nan        half = stats.t.ppf(0.975, n - 1) * sd / np.sqrt(n) if n > 1 else np.nan        out.append({"cohort": c, "model": m, "metric": met, "n_fold": n,                    "mean": v.mean(), "sd": sd, "ci_half": half,                    "ci_lo": v.mean() - half, "ci_hi": v.mean() + half,                    **{f"fold{i}": x for i, x in enumerate(v)}})    d = pd.DataFrame(out)    d["_o"] = d.model.map({m: i for i, m in enumerate(ORDER)}).fillna(99)    return d.sort_values(["metric", "cohort", "_o"]).drop(columns="_o")def to_tex(summ, metrics, cohorts, path, title):    head = ("% " + title + "\n"            "% Value = 4-fold mean +- 95% CI (standard deviation in parentheses).\n"            "% Intervals use the t distribution across folds, df=3, t=3.182; the unit is the fold.\n"            "% Intervals are per model; use a paired test to compare models.\n")    cols = [(c, m) for c in cohorts for m in metrics]    lines = [head, r"\begin{tabular}{l" + "c" * len(cols) + "}", r"\toprule"]    lines.append("& " + " & ".join(r"\multicolumn{%d}{c}{%s}" % (len(metrics), c)                                   for c in cohorts) + r"\\")    st = 2    cm = []    for _ in cohorts:        cm.append(r"\cmidrule(lr){%d-%d}" % (st, st + len(metrics) - 1))        st += len(metrics)    lines.append("".join(cm))    lines.append("Method & " + " & ".join(PRETTY.get(m, m) for _, m in cols) + r"\\")    lines.append(r"\midrule")    models = [m for m in ORDER if m in set(summ.model)]    for m in models:        cells = []        for c, met in cols:            r = summ[(summ.cohort == c) & (summ.model == m) & (summ.metric == met)]            cells.append("--" if not len(r) else                         r"%.3f\,{\tiny$\pm$}%.3f\,{\tiny(%.3f)}"                         % (r["mean"].iloc[0], r.ci_half.iloc[0], r.sd.iloc[0]))        nm = r"\textbf{MoSPR (ours)}" if m == "MoSPR (ours)" else m        lines.append(f"{nm} & " + " & ".join(cells) + r"\\")    lines += [r"\bottomrule", r"\end{tabular}", ""]    path.write_text("\n".join(lines))def _num(x, digits=3):    if x != x:        return "--"    t = f"{x:.{digits}f}"    if t.startswith("0."):        return t[1:]    if t.startswith("-0."):        return "-" + t[2:]    return tdef to_tex_full(summ, metrics, cohorts, path, title):    models = [m for m in ORDER if m in set(summ.model)]    L = [        "% " + title,        r"% Cell = 4-fold mean {\tiny$\pm$95\% CI} {\tiny(SD)}. Leading zeros omitted.",        r"% Intervals use the t distribution across folds (df=3, t=3.182); the unit is the fold.",        r"% Intervals are per model; use a paired test to compare models.",        r"% Requires \usepackage{rotating,booktabs}.",        r"\begin{sidewaystable}[p]",        r"\centering",        r"\tiny",        r"\setlength{\tabcolsep}{2.2pt}",        r"\renewcommand{\arraystretch}{1.25}",        r"\begin{tabular}{l" + "c" * len(models) + "}",        r"\toprule",        "Metric & " + " & ".join(m.replace(" (ours)", "") for m in models) + r"\\",    ]    for c in cohorts:        L.append(r"\midrule")        L.append(r"\multicolumn{%d}{l}{\textbf{%s}}\\" % (len(models) + 1, c))        for met in metrics:            sub = summ[(summ.cohort == c) & (summ.metric == met)].set_index("model")            best = sub["mean"].max() if len(sub) else np.nan            cells = []            for m in models:                if m not in sub.index:                    cells.append("--")                    continue                r = sub.loc[m]                v = _num(r["mean"])                if abs(r["mean"] - best) < 1e-9:                    v = r"\textbf{%s}" % v                cells.append(r"%s\,{\tiny$\pm$%s}\,{\tiny(%s)}"                             % (v, _num(r.ci_half), _num(r.sd)))            L.append(r"\quad %s & " % PRETTY.get(met, met) + " & ".join(cells) + r"\\")    L += [        r"\bottomrule",        r"\end{tabular}",        r"\caption{Per-cohort performance across the four folds. Each cell gives the "        r"mean, the half-width of the 95\% confidence interval across folds "        r"(Student $t$, $df=3$), and the standard deviation in parentheses; leading "        r"zeros are omitted. Gene-level PCC and SCC are correlations across slides "        r"computed per gene and averaged over genes. Pathway metrics are the median "        r"over gene sets of the per-set Spearman correlation, aggregated from "        r"gene-level predictions (pathway labels are never used for training). "        r"Intervals are computed independently per method, so overlapping intervals "        r"do not imply the absence of a difference: every method sees the same folds "        r"and the same held-out slides, and method-to-method comparisons should be "        r"paired. Best value per row in bold.}",        r"\label{tab:appendix-full}",        r"\end{sidewaystable}",        "",    ]    path.write_text("\n".join(L))def main(a):    OUT.mkdir(parents=True, exist_ok=True)    long = load_long(a.cohorts)    long.to_csv(OUT / "stats_foldwise_long.csv", index=False)    summ = summarise(long)    summ.round(4).to_csv(OUT / "stats_summary.csv", index=False)    metrics = ["gene PCC", "gene SCC"] + [f"{c} SCC (median)" for c in COLLECTIONS]    to_tex(summ, ["gene PCC", "gene SCC"], a.cohorts,           OUT / "appendix_gene_stats.tex",           "Appendix - gene level, method x cohort")    to_tex(summ, [f"{c} SCC (median)" for c in COLLECTIONS], a.cohorts,           OUT / "appendix_pathway_stats.tex",           "Appendix - pathway level (median per-set Spearman)")    to_tex_full(summ, metrics, a.cohorts, OUT / "appendix_full_transposed.tex",                "Appendix, transposed - rows = cohort x metric, columns = method")    raw = long.pivot_table(index=["metric", "cohort", "model"], columns="fold",                           values="value").round(4)    raw.to_csv(OUT / "stats_foldwise_wide.csv")    print(f"models {summ.model.nunique()}, cohorts {summ.cohort.nunique()}, "          f"metrics {summ.metric.nunique()}, cells {len(summ)}")    miss = summ[summ.n_fold != 4]    print("cells without 4 folds:", "none" if not len(miss) else          miss[["cohort", "model", "metric", "n_fold"]].to_string(index=False))    print(f"\nwritten: {OUT}/stats_summary.csv · stats_foldwise_{{long,wide}}.csv · "          f"appendix_{{gene,pathway}}_stats.tex · appendix_full_transposed.tex")

In [ ]:
main(ns(cohorts=["BRCA","KIRC","LUAD"]))